In [10]:
from dotenv import load_dotenv
load_dotenv(override=True)
import os

In [14]:
from openai import OpenAI
ai_client = OpenAI()
ai_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPEN_ROUTER_API_KEY"),
)

In [15]:
def llm(prompt):
    response = ai_client.responses.create(
        model="nvidia/nemotron-3-nano-30b-a3b:free",     
        input=prompt,
    )
    return response.output_text

In [16]:
llm("Hello, how are you?")

'Hello! I’mdoing great, thank you for asking. How can I help you today?'

## Getting FAQ data

In [17]:
import requests

In [ ]:
# get list of courses faq pages
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [22]:
#get courses faq
documents = []
url_prefix = "https://datatalks.club/faq"
for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    documents.extend(course_data)

In [23]:
documents[:5]

[{'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."},
 {'id': 'bfafa427b3',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What are the prerequisites for this course?',
  'answer': 'To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful 

## Search

In [24]:
from minsearch import Index

index = Index(
    text_fields = ["question", "section", "answer"],
    keyword_fields = ["course"]
)

In [25]:
index.fit(documents)

In [27]:
question = "I just discovered the course, can I still join?"

In [37]:
search_results = index.search(question, 
        boost_dict={"question": 2, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"},
        num_results=5)

In [ ]:
def search(question, course="llm-zoomcamp"):

    boost_dict={"question": 2, "section": 0.5},
    filter_dict={"course": course},

    return index.search(question, 
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5)

In [36]:
def rag(question):
    search_results = search(question)
    user_prompt = ""
    return llm(user_prompt)